It reads the original laser-power CSV files directly from the repository, performs the analysis, and writes publishable results back into the project workspace.

- The original power measurement raw CSV files are uploaded to `data/<microscope>` folder.
- The optional `target_month.txt` file has the input of the month (`YYYY-MM`) used to calculate the 70% out-of-spec threshold. When it is not set, each wavelength uses its most recent available month.

::: {.callout-note}
The expected filename format is `MM-YY_WAVELENGTH.csv`, such as `02-26_405.csv`.
:::

In [ ]:
#| label: setup
#| output: false

from __future__ import annotations

import os
import re
import io
from contextlib import redirect_stdout
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_DIR = Path(
    os.environ.get("QUARTO_PROJECT_DIR", ".")
).resolve()

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_ROOT = PROJECT_DIR / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MICROSCOPE_DIRS = sorted(
    [
        folder
        for folder in DATA_DIR.iterdir()
        if folder.is_dir() and not folder.name.startswith(".")
    ],
    key=lambda folder: folder.name.lower(),
)

configuration = pd.DataFrame(
    {
        "Setting": [
            "Project directory",
            "Data directory",
            "Output directory",
            "Microscopes detected",
        ],
        "Value": [
            str(PROJECT_DIR),
            str(DATA_DIR),
            str(OUTPUT_ROOT),
            ", ".join(folder.name for folder in MICROSCOPE_DIRS),
        ],
    }
)

display(configuration)

In [ ]:
#| label: parsing-functions

FILENAME_PATTERN = re.compile(
    r"^(?P<month>\d{2})-(?P<year>\d{2}|\d{4})_(?P<wavelength>\d+)\.csv$",
    flags=re.IGNORECASE,
)


def parse_measurement_filename(file_path: Path) -> dict[str, Any]:
    """Parse a filename such as 02-26_405.csv into date and wavelength fields."""
    match = FILENAME_PATTERN.match(file_path.name)
    if not match:
        raise ValueError(
            f"Unexpected filename '{file_path.name}'. Expected MM-YY_WAVELENGTH.csv."
        )

    month = int(match.group("month"))
    year_text = match.group("year")
    year = int(year_text)
    if len(year_text) == 2:
        # The QA archive uses 21st-century two-digit years (for example, 25 and 26).
        year += 2000

    try:
        measurement_date = pd.Timestamp(year=year, month=month, day=1)
    except ValueError as exc:
        raise ValueError(f"Invalid month/year in '{file_path.name}'.") from exc

    return {
        "path": file_path,
        "wavelength": match.group("wavelength"),
        "date": measurement_date,
        "month": measurement_date.strftime("%Y-%m"),
        "legend_label": measurement_date.strftime("%b %Y"),
    }


def read_semicolon_section(file_path: Path, section_names: tuple[str, ...]) -> pd.DataFrame:
    """Read one semicolon-delimited section from a laser-power CSV export."""
    lines = file_path.read_text(encoding="utf-8-sig", errors="replace").splitlines()
    section_names_lower = tuple(name.lower() for name in section_names)

    marker_index = None
    for index, line in enumerate(lines):
        line_lower = line.strip().lower()
        if any(name in line_lower for name in section_names_lower):
            marker_index = index
            break

    if marker_index is None:
        expected = " or ".join(repr(name) for name in section_names)
        raise ValueError(f"No {expected} section found in {file_path.name}.")

    header_index = None
    for index in range(marker_index + 1, len(lines)):
        if lines[index].strip():
            header_index = index
            break

    if header_index is None:
        raise ValueError(f"No header found after the section marker in {file_path.name}.")

    header = [value.strip() for value in lines[header_index].split(";")]
    rows: list[list[str]] = []

    for line in lines[header_index + 1 :]:
        stripped = line.strip()
        if not stripped or stripped.lower().startswith("time"):
            break

        values = [value.strip() for value in stripped.split(";")]
        if len(values) < len(header):
            values.extend([""] * (len(header) - len(values)))
        elif len(values) > len(header):
            values = values[: len(header)]
        rows.append(values)

    if not rows:
        raise ValueError(f"The requested section in {file_path.name} contains no data rows.")

    return pd.DataFrame(rows, columns=header)


def convert_numeric_columns(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Convert columns to numeric when every nonblank value can be converted."""
    result = dataframe.copy()
    for column in result.columns:
        text = result[column].astype(str).str.strip()
        numeric = pd.to_numeric(text.str.replace(",", ".", regex=False), errors="coerce")
        nonblank = text.ne("")
        if nonblank.any() and numeric[nonblank].notna().all():
            result[column] = numeric
    return result


def read_power_instruction_table(file_path: Path) -> pd.DataFrame:
    """Read the Result table values section used for calibration curves."""
    dataframe = read_semicolon_section(file_path, ("Result table values",))
    dataframe.columns = [column.strip() for column in dataframe.columns]
    dataframe = dataframe.rename(
        columns={"power_instruction": "power_percentage_values"}
    )

    if "power_percentage_values" not in dataframe.columns:
        raise ValueError(
            f"The Result table values section in {file_path.name} does not contain "
            "a 'power_instruction' column."
        )

    return convert_numeric_columns(dataframe)


def read_maximum_power(file_path: Path) -> float | None:
    """Read the maximum-power value from Result values or Primary metrics values."""
    try:
        dataframe = read_semicolon_section(
            file_path, ("Result values", "Primary metrics values")
        )
    except ValueError as exc:
        print(f"Warning: {exc} Maximum-power analysis skipped for this file.")
        return None

    dataframe.columns = [column.strip() for column in dataframe.columns]

    # Prefer a clearly named maximum-power column when one exists.
    named_candidates = [
        column
        for column in dataframe.columns
        if "power" in column.lower()
        and ("max" in column.lower() or "maximum" in column.lower())
    ]

    candidate_series = []
    if named_candidates:
        candidate_series.append(dataframe[named_candidates[0]])
    if dataframe.shape[1] >= 3:
        # Retains compatibility with the original notebook, which used column 3.
        candidate_series.append(dataframe.iloc[:, 2])

    for series in candidate_series:
        numeric = pd.to_numeric(
            series.astype(str).str.strip().str.replace(",", ".", regex=False),
            errors="coerce",
        ).dropna()
        if not numeric.empty:
            return float(numeric.iloc[0])

    print(f"Warning: No numeric maximum-power value found in {file_path.name}.")
    return None

In [ ]:
#| label: combination-and-plotting-functions


def combine_calibration_tables(
    records: list[tuple[dict[str, Any], pd.DataFrame]],
) -> pd.DataFrame:
    """Merge monthly calibration tables for one wavelength."""
    combined: pd.DataFrame | None = None
    seen_months: set[str] = set()

    for metadata, dataframe in sorted(records, key=lambda item: item[0]["date"]):
        month = metadata["month"]
        if month in seen_months:
            raise ValueError(
                f"More than one calibration file was found for wavelength "
                f"{metadata['wavelength']} nm in {month}."
            )
        seen_months.add(month)

        other_columns = [
            column for column in dataframe.columns if column != "power_percentage_values"
        ]
        renamed = dataframe.rename(
            columns={column: f"{month}_{column}" for column in other_columns}
        ).drop_duplicates(subset=["power_percentage_values"])

        if combined is None:
            combined = renamed
        else:
            combined = pd.merge(
                combined,
                renamed,
                on="power_percentage_values",
                how="outer",
                validate="one_to_one",
            )

    if combined is None:
        return pd.DataFrame()

    return (
        combined.drop_duplicates(subset=["power_percentage_values"])
        .sort_values("power_percentage_values")
        .reset_index(drop=True)
    )

def plot_calibration_curves(
    combined_all: dict[str, pd.DataFrame], save_folder: Path
) -> list[Path]:
    """Create one calibration-curve plot per wavelength."""
    import re

    save_folder.mkdir(parents=True, exist_ok=True)
    generated: list[Path] = []

    for wavelength in sorted(combined_all, key=int):
        dataframe = combined_all[wavelength]

        # 1. Dynamically scan all columns to extract unique YYYY-MM prefixes matching this wavelength
        extracted_months = set()
        for column in dataframe.columns:
            # Matches any column starting with YYYY-MM followed by this wavelength (e.g., '2025-08_405')
            if f"_{wavelength}" in column:
                match = re.match(r"^(\d{4}-\d{2})", column)
                if match:
                    extracted_months.add(match.group(1))

        # 2. Sort the collected months chronologically
        month_keys = sorted(
            extracted_months,
            key=lambda m: pd.Timestamp(f"{m}-01"),
        )

        if not month_keys:
            print(f"Warning: No columns found for wavelength {wavelength} nm.")
            continue

        figure, axis = plt.subplots(figsize=(8.5, 6))
        plotted_series = 0

        for month in month_keys:
            # 3. Dynamically resolve column names using your layout patterns
            power_column = f"{month}_{wavelength}"
            error_column = f"{month}_error"

            # Fallback check: if the column names contain trailing descriptors, adjust
            if power_column not in dataframe.columns:
                # Find any column that starts with the month and matches the wavelength
                matched_cols = [
                    c
                    for c in dataframe.columns
                    if c.startswith(month) and f"_{wavelength}" in c
                ]
                if matched_cols:
                    power_column = matched_cols[0]
                else:
                    continue

            plot_data = pd.DataFrame(
                {
                    "power_percentage_values": pd.to_numeric(
                        dataframe["power_percentage_values"], errors="coerce"
                    ),
                    "power": pd.to_numeric(dataframe[power_column], errors="coerce"),
                }
            )

            # Check if an accompanying error column exists for this specific month
            error_cols = [
                c
                for c in dataframe.columns
                if c.startswith(month) and "error" in c.lower()
            ]
            if error_cols:
                plot_data["error"] = pd.to_numeric(
                    dataframe[error_cols[0]], errors="coerce"
                ).abs()

            plot_data = plot_data.dropna(
                subset=["power_percentage_values", "power"]
            ).sort_values("power_percentage_values")

            if plot_data.empty:
                continue

            y_error = plot_data["error"] if "error" in plot_data.columns else None

            axis.errorbar(
                plot_data["power_percentage_values"],
                plot_data["power"],
                yerr=y_error,
                marker="o",
                capsize=3,
                linestyle="-",
                elinewidth=1,
                capthick=1,
                label=pd.Timestamp(f"{month}-01").strftime("%b %Y"),
            )
            plotted_series += 1

        if plotted_series == 0:
            plt.close(figure)
            continue

        axis.set_title(f"Laser Power Calibration - {wavelength} nm")
        axis.set_xlabel("Power Percentage Value")
        axis.set_ylabel("Measured Power (µW)")
        axis.grid(True, linestyle="--", alpha=0.5)
        axis.legend(title="Measurement Month")

        figure.tight_layout()
        plot_path = save_folder / f"laser_power_{wavelength}nm.png"
        figure.savefig(plot_path, dpi=300, bbox_inches="tight")
        plt.close(figure)
        generated.append(plot_path)
        print(f"Saved calibration plot: {plot_path.relative_to(PROJECT_DIR)}")

    return generated

def plot_maximum_power_trends(
    maximum_power: pd.DataFrame,
    save_folder: Path,
    target_month: str | None,
) -> tuple[list[Path], pd.DataFrame]:
    """Plot maximum-power trends and calculate wavelength-specific thresholds."""
    generated: list[Path] = []
    threshold_rows: list[dict[str, Any]] = []

    if maximum_power.empty:
        return generated, pd.DataFrame()

    for wavelength, group in maximum_power.groupby("wavelength", sort=False):
        group = group.sort_values("date").reset_index(drop=True)
        reference_month = target_month or group.iloc[-1]["month"]
        reference_rows = group.loc[group["month"] == reference_month]

        threshold_value: float | None = None
        reference_maximum: float | None = None
        if not reference_rows.empty:
            reference_maximum = float(reference_rows.iloc[-1]["power_maximum"])
            threshold_value = reference_maximum * 0.70
        else:
            print(
                f"Warning: target month {reference_month} is not available for "
                f"{wavelength} nm; no threshold line will be drawn."
            )

        threshold_rows.append(
            {
                "wavelength_nm": int(wavelength),
                "reference_month": reference_month,
                "reference_maximum_mW": reference_maximum,
                "out_of_spec_threshold_mW": threshold_value,
            }
        )

        figure, axis = plt.subplots(figsize=(8.5, 6))
        x_labels = group["date"].dt.strftime("%b %Y")
        y_values = pd.to_numeric(group["power_maximum"], errors="coerce")
        axis.plot(x_labels, y_values, marker="s", linestyle="--")

        if threshold_value is not None:
            axis.axhline(
                y=threshold_value,
                linestyle="--",
                label=(
                    f"Out-of-spec threshold: {threshold_value:.2f} mW "
                    f"(70% of {reference_month})"
                ),
            )
            axis.legend()

        finite_values = y_values.dropna().tolist()
        if threshold_value is not None:
            finite_values.append(threshold_value)
        if finite_values:
            minimum = min(finite_values)
            maximum = max(finite_values)
            spread = maximum - minimum
            padding = max(spread * 0.12, abs(maximum) * 0.05, 0.05)
            axis.set_ylim(max(0, minimum - padding), maximum + padding)

        axis.set_title(f"Laser Power Maximum - {wavelength} nm")
        axis.set_xlabel("Measurement Month")
        axis.set_ylabel("Measured Maximum Power (mW)")
        axis.tick_params(axis="x", rotation=45)
        axis.grid(True, linestyle="--", alpha=0.5)
        figure.tight_layout()

        plot_path = save_folder / f"laser_power_max_{wavelength}nm.png"
        figure.savefig(plot_path, dpi=300, bbox_inches="tight")
        plt.close(figure)
        generated.append(plot_path)
        print(f"Saved maximum-power plot: {plot_path.relative_to(PROJECT_DIR)}")

    threshold_summary = pd.DataFrame(threshold_rows).sort_values("wavelength_nm")
    return generated, threshold_summary

In [ ]:
#| label: workflow-functions


def write_excel_workbook(
    combined_all: dict[str, pd.DataFrame],
    maximum_power: pd.DataFrame,
    threshold_summary: pd.DataFrame,
    output_path: Path,
) -> None:
    """Write calibration, maximum-power, and threshold tables to one workbook."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for wavelength in sorted(combined_all, key=int):
            combined_all[wavelength].to_excel(
                writer, sheet_name=f"{wavelength}nm", index=False
            )

        if not maximum_power.empty:
            maximum_power.loc[
                :, ["month", "wavelength", "power_maximum", "source_file"]
            ].to_excel(writer, sheet_name="Maximum Power", index=False)

        if not threshold_summary.empty:
            threshold_summary.to_excel(
                writer, sheet_name="Threshold Summary", index=False
            )

    # Apply compact, readable workbook formatting.
    from openpyxl import load_workbook

    workbook = load_workbook(output_path)
    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        for column_cells in worksheet.columns:
            values = [str(cell.value) if cell.value is not None else "" for cell in column_cells]
            width = min(max(max((len(value) for value in values), default=0) + 2, 12), 36)
            worksheet.column_dimensions[column_cells[0].column_letter].width = width
    workbook.save(output_path)


def run_analysis(
    input_folder: Path,
    output_excel: Path,
    plot_folder: Path,
    target_month: str | None = None,
) -> dict[str, Any]:
    """Run the complete laser-power QA workflow."""
    if target_month is not None:
        try:
            pd.Period(target_month, freq="M")
        except ValueError as exc:
            raise ValueError(
                "TARGET_MONTH must use YYYY-MM format, for example 2026-02."
            ) from exc

    candidate_files = sorted(input_folder.glob("*.csv"))
    parsed_files: list[dict[str, Any]] = []
    ignored_files: list[str] = []

    for file_path in candidate_files:
        try:
            parsed_files.append(parse_measurement_filename(file_path))
        except ValueError as exc:
            ignored_files.append(file_path.name)
            print(f"Warning: {exc} File ignored.")

    if not parsed_files:
        return {
            "status": "no_data",
            "message": (
                f"No valid CSV files were found in {input_folder}. Add files named "
                "MM-YY_WAVELENGTH.csv to the data directory."
            ),
            "ignored_files": ignored_files,
            "plots": [],
        }

    instruction_records: dict[
        str, list[tuple[dict[str, Any], pd.DataFrame]]
    ] = {}
    maximum_rows: list[dict[str, Any]] = []
    skipped_instruction_files: list[str] = []

    for metadata in parsed_files:
        file_path = metadata["path"]
        try:
            instruction_table = read_power_instruction_table(file_path)
            instruction_records.setdefault(metadata["wavelength"], []).append(
                (metadata, instruction_table)
            )
        except ValueError as exc:
            skipped_instruction_files.append(file_path.name)
            print(f"Warning: {exc} Calibration-curve analysis skipped for this file.")

        maximum_value = read_maximum_power(file_path)
        if maximum_value is not None:
            maximum_rows.append(
                {
                    "date": metadata["date"],
                    "month": metadata["month"],
                    "wavelength": int(metadata["wavelength"]),
                    "power_maximum": maximum_value,
                    "source_file": file_path.name,
                }
            )

    combined_all = {
        wavelength: combine_calibration_tables(records)
        for wavelength, records in instruction_records.items()
        if records
    }

    maximum_power = pd.DataFrame(maximum_rows)
    if not maximum_power.empty:
        maximum_power = maximum_power.sort_values(
            ["wavelength", "date", "source_file"]
        ).reset_index(drop=True)

    calibration_plots = plot_calibration_curves(combined_all, plot_folder)
    maximum_plots, threshold_summary = plot_maximum_power_trends(
        maximum_power, plot_folder, target_month
    )

    if combined_all or not maximum_power.empty or not threshold_summary.empty:
        write_excel_workbook(
            combined_all, maximum_power, threshold_summary, output_excel
        )
        print(f"Saved Excel workbook: {output_excel.relative_to(PROJECT_DIR)}")

    file_summary = pd.DataFrame(
        [
            {
                "source_file": metadata["path"].name,
                "month": metadata["month"],
                "wavelength_nm": int(metadata["wavelength"]),
            }
            for metadata in parsed_files
        ]
    ).sort_values(["month", "wavelength_nm"])

    return {
        "status": "ok",
        "message": f"Processed {len(parsed_files)} valid CSV files.",
        "file_summary": file_summary,
        "combined_all": combined_all,
        "maximum_power": maximum_power,
        "threshold_summary": threshold_summary,
        "ignored_files": ignored_files,
        "skipped_instruction_files": skipped_instruction_files,
        "plots": calibration_plots + maximum_plots,
        "excel_path": output_excel,
    }
def get_target_month(microscope_dir: Path) -> str | None:
    """Read a microscope-specific target month from target_month.txt."""

    target_file = microscope_dir / "target_month.txt"

    if not target_file.exists():
        return None

    target_month = target_file.read_text(
        encoding="utf-8"
    ).strip()

    if not target_month:
        return None

    try:
        pd.Period(target_month, freq="M")
    except ValueError as exc:
        raise ValueError(
            f"{microscope_dir.name}/target_month.txt must contain "
            f"a month in YYYY-MM format. Found: {target_month}"
        ) from exc

    return target_month

def write_microscope_report(
    microscope: str,
    results: dict,
    output_dir: Path,
) -> Path:
    """Create Markdown content for one microscope page."""

    report_path = output_dir / "report.md"
    microscope_title = microscope.replace("_", " ")

    lines = []

    if results["status"] != "ok":
        lines.append(
            f"No valid laser-power data are available for **{microscope_title}**."
        )
        report_path.write_text("\n".join(lines), encoding="utf-8")
        return report_path

    # Optional analysis summary
    lines.append(f"**{results['message']}**")
    lines.append("")

    # Calibration plots
    calibration_plots = [
        plot
        for plot in results["plots"]
        if not plot.stem.startswith("laser_power_max_")
    ]

    if calibration_plots:
        lines.append("## Laser Power Calibration")
        lines.append("")

        for plot_path in calibration_plots:
            match = re.search(
                r"^laser_power_(\d+)nm$",
                plot_path.stem,
            )

            if match:
                wavelength = match.group(1)
                title = f"Laser Power - {wavelength} nm"
            else:
                title = plot_path.stem.replace("_", " ").title()

            lines.append(f"### {title}")
            lines.append("")

            image_path = (
                f"../outputs/{microscope}/plots/{plot_path.name}"
            )

            lines.append(
                f"![{title}]({image_path})"
            )
            lines.append("")

    # Maximum-power plots
    maximum_plots = [
        plot
        for plot in results["plots"]
        if plot.stem.startswith("laser_power_max_")
    ]

    if maximum_plots:
        lines.append("## Maximum Laser Power")
        lines.append("")

        for plot_path in maximum_plots:
            match = re.search(
                r"^laser_power_max_(\d+)nm$",
                plot_path.stem,
            )

            if match:
                wavelength = match.group(1)
                title = f"Maximum Laser Power - {wavelength} nm"
            else:
                title = plot_path.stem.replace("_", " ").title()

            lines.append(f"### {title}")
            lines.append("")

            image_path = (
                f"../outputs/{microscope}/plots/{plot_path.name}"
            )

            lines.append(
                f"![{title}]({image_path})"
            )
            lines.append("")

    report_path.write_text(
        "\n".join(lines),
        encoding="utf-8",
    )

    return report_path
    

In [ ]:
#| label: run-analysis
#| echo: false

import io
from contextlib import redirect_stdout

all_results = {}

for microscope_dir in MICROSCOPE_DIRS:

    microscope = microscope_dir.name

    output_dir = OUTPUT_ROOT / microscope
    plot_dir = output_dir / "plots"
    excel_path = output_dir / "combined_power_data.xlsx"

    output_dir.mkdir(parents=True, exist_ok=True)
    plot_dir.mkdir(parents=True, exist_ok=True)

    # Remove old generated files
    for old_plot in plot_dir.glob("*.png"):
        old_plot.unlink()

    if excel_path.exists():
        excel_path.unlink()

    # Get the target month for THIS microscope
    microscope_target_month = get_target_month(
        microscope_dir
    )

    # Run analysis for THIS microscope
    with redirect_stdout(io.StringIO()):
        results = run_analysis(
            input_folder=microscope_dir,
            output_excel=excel_path,
            plot_folder=plot_dir,
            target_month=microscope_target_month,
        )

    results["microscope"] = microscope
    results["target_month"] = microscope_target_month
    results["output_dir"] = output_dir
    results["excel_path"] = excel_path

    all_results[microscope] = results

## Confocal Microscopes Laser Power Measurements and Plots

In [ ]:
#| echo: false

if all_results:

    display(
        Markdown(
            "Select a microscope to view its laser-power "
            "quality assurance results."
        )
    )

    for microscope in sorted(all_results):

        microscope_title = microscope.replace("_", " ")

        display(
            Markdown(
                f"- [{microscope_title}]"
                f"(microscopes/{microscope}.html)"
            )
        )